# Notebook 03 — Stationarity & Cointegration Analysis

## Improvements Over Original Paper

| Test | Original Paper | This Notebook |
|------|---------------|---------------|
| Unit root | ADF only | ADF + KPSS + Zivot-Andrews |
| Pair relationships | Correlation only | Engle-Granger + Johansen cointegration |
| Structural breaks | Noted visually | Bai-Perron formal detection |
| Weather causality | Visual overlay | Granger causality tests |


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.data.fetch_commodity_data import load_or_fetch, compute_returns
from src.data.fetch_weather_data import load_all_weather, align_weather_to_prices
from src.analysis.stationarity import full_stationarity_report, bai_perron_breaks
from src.analysis.cointegration import (
    run_all_cointegration_tests,
    run_all_granger_tests,
    correlation_report
)

In [ ]:
# Load data
prices  = load_or_fetch()
returns = compute_returns(prices, freq='D')
print(f'Price shape:  {prices.shape}')
print(f'Return shape: {returns.shape}')

In [ ]:
# ── STATIONARITY REPORT ────────────────────────────────────────────
# Original paper: ADF only
# Improvement: ADF + KPSS + Zivot-Andrews

print('PRICE LEVELS:')
price_report = full_stationarity_report(prices, series_type='prices')
display(price_report)

In [ ]:
print('DAILY RETURNS:')
return_report = full_stationarity_report(returns, series_type='returns')
display(return_report)

In [ ]:
# ── STRUCTURAL BREAK DETECTION ─────────────────────────────────────
# Improvement: Formally identifies break dates (e.g. Silver Thursday 1980)

print('Structural Break Detection — Silver (example)')
silver_breaks = bai_perron_breaks(prices['Silver'], max_breaks=5)

print(f"  Identified {silver_breaks['n_breaks']} break(s)")
print(f"  Break dates: {silver_breaks['break_dates']}")
print('  Segment statistics:')
for seg in silver_breaks['segments']:
    print(f"    {seg['start']} → {seg['end']}: mean={seg['mean']}, std={seg['std']}")

In [ ]:
# ── CORRELATION ANALYSIS (replicates original paper Figure 4.2) ────
from src.analysis.cointegration import correlation_report

daily_corr   = correlation_report(returns, 'Daily')

weekly_ret   = compute_returns(prices, freq='W')
weekly_corr  = correlation_report(weekly_ret, 'Weekly')

monthly_ret  = compute_returns(prices, freq='ME')
monthly_corr = correlation_report(monthly_ret, 'Monthly')

In [ ]:
# ── COINTEGRATION TESTS (NEW) ──────────────────────────────────────
# Major improvement: Engle-Granger + Johansen for all four pairs

coint_summary = run_all_cointegration_tests(prices, use_log=True)
display(coint_summary)

In [ ]:
# ── GRANGER CAUSALITY: WEATHER → PRICES (NEW) ─────────────────────
# Improvement: Replaces visual inspection of weather overlays

# Load weather data (requires prior download in notebook 01)
try:
    weather = load_all_weather()
    weather_aligned = align_weather_to_prices(prices, weather)
    granger_summary = run_all_granger_tests(returns, weather_aligned, max_lags=12)
    display(granger_summary)
except FileNotFoundError:
    print('Weather data not found. Run notebook 01 with NOAA API token first.')